# RAG Pipeline — Document Assistant

## 1. Project Objective

This notebook builds and evaluates the Retrieval-Augmented Generation (RAG) pipeline
that powers the FastAPI + Streamlit application in this repository.

**Goal:** given a small collection of documents (PDF / DOCX / HTML), answer natural-language
questions with answers that are *grounded* in those documents and that cite the source
file (and page, when available) each answer came from.

Pipeline covered in this notebook:

```
Documents -> Load -> Clean -> Chunk -> Embed -> Chroma (persisted) -> Retrieve -> Cloud LLM -> Grounded answer + sources
```

This notebook reuses the **exact same code** as the backend (`api/chroma_utils.py` and
`api/langchain_utils.py`) instead of re-implementing the pipeline separately, so there is
no risk of the notebook and the running application drifting apart.



### Setup

Add the `api/` folder to the path so we can import its modules directly.

In [2]:
import sys
import os
from pathlib import Path

API_DIR = os.path.abspath(os.path.join("..", "backend"))
sys.path.insert(0, API_DIR)

# Run api/ code from the api/ folder so relative paths (./chroma_db, rag_app.db)
# match what the backend uses.
os.chdir(API_DIR)

from dotenv import load_dotenv
load_dotenv(os.path.join(API_DIR, "..", ".env"))

print("Working directory:", os.getcwd())


Working directory: d:\All\iti_tasks\rag-assistant\backend


## 2. Load Documents

We scan `../docs/` for any `.pdf`, `.docx`, or `.html` files. Add your own documents to
that folder (or upload them through the Streamlit sidebar, which writes into the same
persisted Chroma collection) before running this cell.


In [3]:
DOCS_DIR = Path("..") / "docs"
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".html"}

source_files = sorted(
    p for p in DOCS_DIR.glob("**/*")
    if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
)

print(f"Found {len(source_files)} supported document(s) in {DOCS_DIR.resolve()}:")
for p in source_files:
    print(" -", p.name)

if not source_files:
    print(
        "\nNo documents found yet. Add a few PDF/DOCX/HTML files to the docs/ folder "
        "and re-run this cell before continuing."
    )


Found 2 supported document(s) in D:\All\iti_tasks\rag-assistant\docs:
 - Hands_On_Machine_Learning_with_Scikit_Le.pdf
 - test.html


## 3. Inspect Documents

For each file we try to load it with the same loader the backend uses, and report how
many pages/sections it produced and whether it failed to parse (e.g. a scanned PDF with
no extractable text would need OCR, which is out of scope for this project).


In [4]:
from chroma_utils import load_and_split_document

inspection_rows = []
raw_documents_by_file = {}

for path in source_files:
    try:
        # Load without splitting first, just to report page/section counts.
        from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, BSHTMLLoader
        ext = path.suffix.lower()
        if ext == ".pdf":
            loader = PyPDFLoader(str(path))
        elif ext == ".docx":
            loader = Docx2txtLoader(str(path))
        else:
            loader = BSHTMLLoader(str(path), open_encoding="utf-8")
        docs = loader.load()
        raw_documents_by_file[path.name] = docs
        total_chars = sum(len(d.page_content) for d in docs)
        inspection_rows.append({
            "filename": path.name,
            "format": ext,
            "pages_or_sections": len(docs),
            "total_characters": total_chars,
            "status": "OK" if total_chars > 0 else "EMPTY - may need OCR",
        })
    except Exception as exc:
        inspection_rows.append({
            "filename": path.name,
            "format": path.suffix.lower(),
            "pages_or_sections": 0,
            "total_characters": 0,
            "status": f"FAILED TO PARSE: {exc}",
        })

import pandas as pd
inspection_df = pd.DataFrame(inspection_rows)
inspection_df


d:\All\iti_tasks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\All\iti_tasks\rag-assistant\backend\chroma_utils.py:55: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL_NAME)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3017.32it/s]


,filename,format,pages_or_sections,total_characters,status
0,Hands_On_Machine_Learning_with_Scikit_Le.pdf,.pdf,1150,1668200,OK
1,test.html,.html,0,0,FAILED TO PARSE: By default BSHTMLLoader uses ...


## 4. Basic Cleaning

`chroma_utils._clean_text` collapses repeated whitespace/blank lines and strips stray
null bytes that sometimes appear in PDF extractions. It's intentionally light-touch: we
don't want to accidentally remove real content.

In [5]:
from chroma_utils import _clean_text

if raw_documents_by_file:
    sample_filename, sample_docs = next(iter(raw_documents_by_file.items()))
    before = sample_docs[0].page_content
    after = _clean_text(before)
    print(f"Example cleaning on the first page/section of '{sample_filename}':\n")
    print("BEFORE (first 300 chars):\n", repr(before[:300]))
    print("\nAFTER (first 300 chars):\n", repr(after[:300]))
else:
    print("No documents loaded yet - see Section 2.")


Example cleaning on the first page/section of 'Hands_On_Machine_Learning_with_Scikit_Le.pdf':

BEFORE (first 300 chars):
 ''

AFTER (first 300 chars):
 ''


## 5. Chunking

## 6. Chunk Size & Overlap — Why 1800 / 350?

We use `RecursiveCharacterTextSplitter(chunk_size=1800, chunk_overlap=350)`, the same
values as the backend (`api/chroma_utils.py`).

- **`chunk_size=1800` characters (roughly 280-320 words):** the source document here is a
  long, section-based technical book, not a short spec sheet or FAQ entry. A single ML
  concept (e.g. explaining gradient descent, or a code example plus the paragraph that
  introduces it) is usually spread across several paragraphs. The earlier default of 1000
  characters routinely cut that explanation in half, so the LLM only ever saw part of it.
  1800 characters keeps a whole explanation - and often a short code block with it - inside
  one chunk, at the cost of each chunk covering a slightly broader topic.
- **`chunk_overlap=350` characters (~20%):** a bigger chunk size still risks cutting a
  sentence or a code block at the boundary, so the overlap was increased proportionally to
  keep that protection.
- **Custom `separators`:** the splitter is told to prefer breaking on markdown-style
  headings and paragraph breaks before falling back to a hard character cut, so it doesn't
  split a heading from the paragraph underneath it or slice a code block down the middle.

These values are tuned for one long, prose-and-code technical book. Short documents (spec
sheets, FAQs) generally do better with the smaller 1000/200 setting we used before -
worth reverting if you go back to that kind of source, and worth comparing both settings
side by side in Section 9/14 if retrieval quality looks weak.

In [6]:
from chroma_utils import text_splitter, CHUNK_SIZE, CHUNK_OVERLAP

print("chunk_size:", CHUNK_SIZE, "| chunk_overlap:", CHUNK_OVERLAP)

all_chunks = []
for path in source_files:
    chunks = load_and_split_document(str(path), filename=path.name)
    all_chunks.extend(chunks)
    print(f"{path.name}: {len(chunks)} chunk(s)")

print(f"\nTotal chunks across all documents: {len(all_chunks)}")
if all_chunks:
    print("\nExample chunk metadata:", all_chunks[0].metadata)
    print("\nExample chunk text (first 300 chars):\n", all_chunks[0].page_content[:300])


chunk_size: 1800 | chunk_overlap: 350
Hands_On_Machine_Learning_with_Scikit_Le.pdf: 1432 chunk(s)
test.html: 1 chunk(s)

Total chunks across all documents: 1433

Example chunk metadata: {'producer': 'calibre (4.8.0) [http://calibre-ebook.com]', 'creator': 'calibre (4.8.0) [http://calibre-ebook.com]', 'creationdate': '2020-02-11T18:06:47+00:00', 'author': 'Aurélien Géron', 'moddate': '2020-02-11T19:08:07+01:00', 'title': 'Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow', 'source': '..\\docs\\Hands_On_Machine_Learning_with_Scikit_Le.pdf', 'total_pages': 1150, 'page': 2, 'page_label': '2', 'filename': 'Hands_On_Machine_Learning_with_Scikit_Le.pdf'}

Example chunk text (first 300 chars):
 Hands-On Machine Learning
with Scikit-Learn, Keras, and
TensorFlow
SECOND EDITION
Concepts, Tools, and Techniques to Build Intelligent
Systems
Aurélien Géron


## 7. Generate Embeddings

Embeddings are generated with `sentence-transformers/all-MiniLM-L6-v2` — small (~90MB,
downloaded once and cached), fast on CPU, and good enough quality for this project's
scale. This is the same `embedding_function` object the backend uses.

In [7]:
from chroma_utils import embedding_function, EMBEDDING_MODEL_NAME

print("Embedding model:", EMBEDDING_MODEL_NAME)

if all_chunks:
    sample_text = all_chunks[0].page_content
    sample_vector = embedding_function.embed_query(sample_text)
    print("Embedding dimensionality:", len(sample_vector))
    print("First 8 values:", sample_vector[:8])


Embedding model: all-MiniLM-L6-v2
Embedding dimensionality: 384
First 8 values: [-0.05393769219517708, -0.059475354850292206, 0.06206726282835007, -0.01878935843706131, 0.06427028775215149, -0.08588618785142899, -0.037983495742082596, -0.03642866760492325]


## 8. Create / Use the Persistent ChromaDB

`chroma_utils.vectorstore` already points at the same `./chroma_db` folder the FastAPI
backend uses (persisted to disk - re-running this notebook or restarting the backend does
**not** lose your indexed documents).

Re-running this cell is safe: `index_document_to_chroma` just adds chunks, so if you run
it twice on the same file you'll get duplicate chunks with the same `file_id`. Use the
Streamlit "Delete Selected Document" button (or `delete_doc_from_chroma`) first if you
want to re-index a file from scratch.

In [8]:
from chroma_utils import vectorstore, index_document_to_chroma

# Use a distinct negative range for notebook-indexed file_ids so they don't collide with
# ids assigned by the running backend's SQLite `document_store` table.
NOTEBOOK_FILE_ID_BASE = -1000

for i, path in enumerate(source_files):
    fake_file_id = NOTEBOOK_FILE_ID_BASE - i
    success = index_document_to_chroma(str(path), file_id=fake_file_id, filename=path.name)
    print(f"{path.name}: {'indexed' if success else 'FAILED'} (file_id={fake_file_id})")

try:
    collection_count = vectorstore._collection.count()
    print(f"\nTotal chunks currently stored in Chroma: {collection_count}")
except Exception as exc:
    print("Could not read collection count:", exc)


Hands_On_Machine_Learning_with_Scikit_Le.pdf: indexed (file_id=-1000)
test.html: indexed (file_id=-1001)

Total chunks currently stored in Chroma: 1433


## 9. Retrieve Relevant Chunks

A quick sanity check: for a single question, what does the retriever actually return?


In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})  # matches api/langchain_utils.py

test_question = "What is this document about?"  # replace with a real question for your domain
retrieved = retriever.invoke(test_question)

print(f"Retrieved {len(retrieved)} chunk(s) for: {test_question!r}\n")
for i, doc in enumerate(retrieved, start=1):
    print(f"[{i}] {doc.metadata.get('filename')} (page {doc.metadata.get('page')})")
    print("    ", doc.page_content[:150].replace("\n", " "), "...")


Retrieved 6 chunk(s) for: 'What is this document about?'

[1] Hands_On_Machine_Learning_with_Scikit_Le.pdf (page 16)
     To comment or ask technical questions about this book, send email to bookquestions@oreilly.com. For more information about our books, courses, confere ...
[2] Hands_On_Machine_Learning_with_Scikit_Le.pdf (page 18)
     Monaghan and Amanda Kersey for their thorough copyediting (respectively for the first and second edition), and to Johnny O’Toole who managed the relat ...
[3] Hands_On_Machine_Learning_with_Scikit_Le.pdf (page 17)
     Wilder-James, and Yuefeng Zhou, all of whom were tremendously helpful. Huge thanks to all of you, and to all other members of the TensorFlow team, not ...
[4] Hands_On_Machine_Learning_with_Scikit_Le.pdf (page 1149)
     About the Author Aurélien Géron is a Machine Learning consultant and lecturer. A former Googler, he led YouTube’s video classification team from 2013  ...
[5] Hands_On_Machine_Learning_with_Scikit_Le.pdf (page 4)
     S

## 10. Build the RAG Prompt

We reuse `api/langchain_utils.py`'s `rag_chain` and `get_answer()` directly, so the
prompt tested here is exactly the prompt the running API uses. It instructs the LLM to
answer only from the retrieved context, never invent facts, and say so explicitly when
the documents don't contain the answer.

> This cell makes a real call to your configured cloud LLM. Make sure `LLM_API_KEY` is
> set in your `.env` file before running it.

In [10]:
from langchain_utils import get_answer, qa_system_prompt

print(qa_system_prompt)


You are a helpful assistant that answers questions about the user's uploaded documents.

Rules you MUST follow:
1. Answer using ONLY the information in the context below.
2. Do not invent, assume, or add facts that are not present in the context.
3. If the context does not contain enough information to answer the question, respond exactly with: "I could not find this information in the provided documents."
4. Keep the answer concise and directly useful.

Context:
{context}


## 11. Ask at Least 10 Sample Questions

## 12. Show Retrieved Sources

Replace the placeholder questions below with ones relevant to your chosen document
domain. Each question is run through the full pipeline (retrieve -> LLM -> answer), and
we print the sources returned alongside each answer.

In [11]:
QUESTIONS = [
    "What is this document about?",
    "Summarize the main topic in two sentences.",
    "What are the key requirements or steps mentioned?",
    "Who is the intended audience for this document?",
    "Are there any dates or deadlines mentioned?",
    "What definitions or key terms are introduced?",
    "What examples are given, if any?",
    "Are there any numeric limits, prices, or quantities mentioned?",
    "What should someone do if something goes wrong, according to the document?",
    "Is there anything explicitly out of scope or excluded?",
    # A question that should NOT be answerable from the documents, used in Section 13
    # to demonstrate that the model admits when it doesn't know rather than hallucinating.
    "What is the capital of Australia?",
]

qa_results = []
chat_history = []  # fresh session for this evaluation run

for question in QUESTIONS:
    try:
        answer, sources = get_answer(question, chat_history)
    except Exception as exc:
        answer, sources = f"ERROR calling the LLM: {exc}", []

    qa_results.append({"question": question, "answer": answer, "sources": sources})

    print(f"Q: {question}")
    print(f"A: {answer}")
    if sources:
        for s in sources:
            page = f", page {s['page']}" if s.get("page") is not None else ""
            print(f"   source: {s['filename']}{page}")
    else:
        print("   source: (none retrieved / not grounded)")
    print()


Q: What is this document about?
A: This document is the front‑matter of the book *Hands‑On Machine Learning with Scikit‑Learn, Keras, and TensorFlow* by Aurélien Géron. It contains the author’s acknowledgments, a brief biography, information about the book’s supplemental code notebooks, and publisher/legal notices. The book itself is a practical guide to machine learning using scikit‑learn, Keras, and TensorFlow.
   source: Hands_On_Machine_Learning_with_Scikit_Le.pdf, page 16
   source: Hands_On_Machine_Learning_with_Scikit_Le.pdf, page 18
   source: Hands_On_Machine_Learning_with_Scikit_Le.pdf, page 17
   source: Hands_On_Machine_Learning_with_Scikit_Le.pdf, page 1149
   source: Hands_On_Machine_Learning_with_Scikit_Le.pdf, page 4
   source: Hands_On_Machine_Learning_with_Scikit_Le.pdf, page 13

Q: Summarize the main topic in two sentences.
A: The passage explains how text data can be represented using a bag‑of‑words approach and TF‑IDF weighting, and discusses the use of Keras prepr

## 13. Demonstrate Citation Grounding

Two things to check in the output above:

1. For questions that **are** covered by your documents, the answer should reference
   information that actually appears in the cited source/page (spot-check a couple by
   opening the source file).
2. For the last question ("What is the capital of Australia?" - deliberately unrelated to
   the uploaded documents), the model should respond with something close to *"I could
   not find this information in the provided documents"* rather than answering from its
   own general knowledge. That's the grounding rule from the system prompt working as
   intended.

If the last question *was* answered instead of refused, see Section 16 (Failure
Analysis) - this usually means the retriever returned irrelevant chunks that the LLM
then over-relied on, or the model isn't following the grounding instruction strictly.

In [12]:
last_result = qa_results[-1]
print("Off-topic question:", last_result["question"])
print("Model's answer:", last_result["answer"])
print(
    "\nGrounded correctly?"
    if "not find" in last_result["answer"].lower()
    else "\nNOT grounded - see Section 16 (Failure Analysis)."
)


Off-topic question: What is the capital of Australia?
Model's answer: I could not find this information in the provided documents.

Grounded correctly?


## 14. Evaluation

## 15. Evaluation Table

The table below is built from the 10+ questions run in Section 11. `Generated Answer`,
`Retrieved Context`, and `Sources` are filled in automatically. `Expected Answer` and
`Correct?` are **placeholders you must fill in by hand** after reading each generated
answer and checking it against your source documents - correctness is a judgment call
only a human reviewer (you) can make. `Grounded?` is a simple automatic heuristic (did
the model return at least one source, or explicitly say it couldn't find the answer?).

In [13]:
eval_rows = []
for r in qa_results:
    grounded = bool(r["sources"]) or "not find" in r["answer"].lower()
    context_preview = "; ".join(
        f"{s['filename']} (p.{s['page']})" if s.get("page") is not None else s["filename"]
        for s in r["sources"]
    ) or "(none retrieved)"

    eval_rows.append({
        "Question": r["question"],
        "Expected Answer": "TODO: fill in after reading the source document(s)",
        "Retrieved Context": context_preview,
        "Generated Answer": r["answer"],
        "Grounded?": "Yes" if grounded else "No",
        "Correct?": "TODO",
        "Sources": context_preview,
    })

evaluation_df = pd.DataFrame(eval_rows)
evaluation_df


,Question,Expected Answer,Retrieved Context,Generated Answer,Grounded?,Correct?,Sources
0,What is this document about?,TODO: fill in after reading the source documen...,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...,This document is the front‑matter of the book ...,Yes,TODO,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...
1,Summarize the main topic in two sentences.,TODO: fill in after reading the source documen...,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...,The passage explains how text data can be repr...,Yes,TODO,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...
2,What are the key requirements or steps mentioned?,TODO: fill in after reading the source documen...,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...,**Key requirements / steps mentioned in the co...,Yes,TODO,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...
3,Who is the intended audience for this document?,TODO: fill in after reading the source documen...,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...,I could not find this information in the provi...,Yes,TODO,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...
4,Are there any dates or deadlines mentioned?,TODO: fill in after reading the source documen...,test.html; Hands_On_Machine_Learning_with_Scik...,"The documents mention time‑based limits (e.g.,...",Yes,TODO,test.html; Hands_On_Machine_Learning_with_Scik...
5,What definitions or key terms are introduced?,TODO: fill in after reading the source documen...,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...,**Definitions / key terms introduced in the pr...,Yes,TODO,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...
6,"What examples are given, if any?",TODO: fill in after reading the source documen...,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...,"The documents list several concrete examples, ...",Yes,TODO,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...
7,"Are there any numeric limits, prices, or quant...",TODO: fill in after reading the source documen...,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...,Yes. The documents mention:\n\n* The median ho...,Yes,TODO,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...
8,What should someone do if something goes wrong...,TODO: fill in after reading the source documen...,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...,"If something goes wrong, the document says to ...",Yes,TODO,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...
9,Is there anything explicitly out of scope or e...,TODO: fill in after reading the source documen...,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...,I could not find this information in the provi...,Yes,TODO,Hands_On_Machine_Learning_with_Scikit_Le.pdf (...


In [14]:
# Save the evaluation table for the README / submission.
evaluation_df.to_csv("../notebooks/evaluation_results.csv", index=False)
print("Saved to notebooks/evaluation_results.csv")


Saved to notebooks/evaluation_results.csv


## 16. Failure Analysis

After filling in `Expected Answer` and `Correct?` above, summarize here what went wrong,
for example:

- *Which questions got an ungrounded or incorrect answer, and why (bad retrieval? chunk
  boundary cut off the answer? question was ambiguous)?*
- *Did increasing `k` (number of retrieved chunks) in `retriever.as_retriever(search_kwargs={"k": ...})`
  help on the questions that failed?*
- *Did any document fail to parse cleanly in Section 3, and did that show up as a
  retrieval gap here?*

**TODO:** replace this paragraph with your own findings once you've run this notebook on
your real document set and filled in the evaluation table.

## 17. Limitations

- Uses a **cloud LLM API** instead of a local Ollama model — see the root README for why,
  and note that answer quality/latency now depend on the chosen provider and model.
- Embeddings are generated with a small, general-purpose model
  (`all-MiniLM-L6-v2`); a larger or domain-specific embedding model could improve
  retrieval quality, at the cost of more disk space and slower embedding.
- Chunking is fixed-size (character-based), not semantic - it can occasionally split a
  logical unit (e.g. a table row, a numbered step) across two chunks.
- Scanned PDFs / images with no extractable text are **not** supported (would need OCR,
  which is out of scope for this project).
- Retrieval uses simple top-k similarity search (`k=4`); no re-ranking or hybrid
  keyword+vector search is implemented.
- Evaluation in this notebook is small-scale (a handful of questions) and manually
  judged - it is not a statistically rigorous benchmark.

## 18. Persist / Export the Vector Store

Chroma with `persist_directory="./chroma_db"` (relative to `api/`) writes to disk
automatically on every `add_documents()` call - there is no separate "save" step. The
cell below just confirms the data is really on disk, and writes out a small config file
recording exactly which chunk size, overlap, and embedding model were used, so the
result is reproducible.

In [15]:
import json as _json

chroma_dir = Path("chroma_db")  # relative to api/, since we os.chdir()'d there earlier
print("Chroma persisted at:", chroma_dir.resolve())
print("Exists on disk:", chroma_dir.exists())
if chroma_dir.exists():
    total_size_mb = sum(f.stat().st_size for f in chroma_dir.rglob("*") if f.is_file()) / (1024 * 1024)
    print(f"On-disk size: {total_size_mb:.2f} MB")

config_snapshot = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "llm_model": DEFAULT_MODEL_NAME if 'DEFAULT_MODEL_NAME' in dir() else os.getenv("LLM_MODEL"),
    "vector_store": "chromadb",
    "persist_directory": str(chroma_dir.resolve()),
}

with open("../notebooks/pipeline_config.json", "w") as f:
    _json.dump(config_snapshot, f, indent=2)

print("\nSaved pipeline config to notebooks/pipeline_config.json:")
print(_json.dumps(config_snapshot, indent=2))


Chroma persisted at: D:\All\iti_tasks\rag-assistant\backend\chroma_db
Exists on disk: True
On-disk size: 24.53 MB

Saved pipeline config to notebooks/pipeline_config.json:
{
  "embedding_model": "all-MiniLM-L6-v2",
  "chunk_size": 1800,
  "chunk_overlap": 350,
  "llm_model": "openai/gpt-oss-20b",
  "vector_store": "chromadb",
  "persist_directory": "D:\\All\\iti_tasks\\rag-assistant\\backend\\chroma_db"
}


---

**End of notebook.** The persisted `api/chroma_db/` folder produced here is exactly what
`api/main.py` loads when the FastAPI backend starts - no rebuilding required.